# 五十音教學網頁　產生器

這份 notebook 會讀取專案資料夾裡的三個來源檔，組成一個完全自包含的 `index.html`。

| 來源檔 | 內容 | 影響網頁的哪裡 |
|---|---|---|
| `五十音單字表_草稿.xlsx` | 單字表、常用短句 | 放大字卡的單字、單字測驗、句型練習、常用短句頁 |
| `50音記憶_整理版.docx` | 字形／發音記憶法 | 放大字卡裡的「💡 記憶法」 |
| `kanjivg_kana.json` | 筆順路徑 | 放大字卡的筆順動畫 |

**使用方式**：改完來源檔 → 從上到下執行所有 cell → `index.html` 就會更新 → commit & push。

**需要的套件**：`pandas`、`openpyxl`、`python-docx`（`pip install pandas openpyxl python-docx`）

> 筆順資料來自 [KanjiVG](http://kanjivg.tagaini.net)（Ulrich Apel，CC BY-SA 3.0），產生的網頁需保留頁尾的出處標示。

## 1. 路徑設定

In [1]:
from pathlib import Path
import json, random
import pandas as pd
from docx import Document as _Doc

# 這份 notebook 請放在專案資料夾（japanese_50）裡再執行
BASE = Path.cwd()
if not (BASE / "五十音單字表_草稿.xlsx").exists() and (BASE.parent / "五十音單字表_草稿.xlsx").exists():
    BASE = BASE.parent

XLSX     = BASE / "五十音單字表_草稿.xlsx"      # 單字表＋常用短句
DOCX     = BASE / "50音記憶_整理版.docx"        # 字形／發音記憶法
KVG_PATH = BASE / "kanjivg_kana.json"          # 筆順路徑資料（KanjiVG，CC BY-SA 3.0）
OUT      = BASE / "index.html"                 # 產生的網頁

for p in (XLSX, DOCX, KVG_PATH):
    print(("OK  " if p.exists() else "缺少 ") + p.name)
print("\n輸出到：", OUT)

OK  五十音單字表_草稿.xlsx
OK  50音記憶_整理版.docx
OK  kanjivg_kana.json

輸出到： /sessions/epic-busy-ride/mnt/japanese_50/index.html


## 2. 五十音基礎資料
清音、濁音、拗音的欄位定義，以及小字假名的縮小處理。

In [2]:
B=""
seion=[
 {"head":"あ行","r":["a","i","u","e","o"],"h":["あ","い","う","え","お"],"k":["ア","イ","ウ","エ","オ"]},
 {"head":"か行","r":["ka","ki","ku","ke","ko"],"h":["か","き","く","け","こ"],"k":["カ","キ","ク","ケ","コ"]},
 {"head":"さ行","r":["sa","shi","su","se","so"],"h":["さ","し","す","せ","そ"],"k":["サ","シ","ス","セ","ソ"]},
 {"head":"た行","r":["ta","chi","tsu","te","to"],"h":["た","ち","つ","て","と"],"k":["タ","チ","ツ","テ","ト"]},
 {"head":"な行","r":["na","ni","nu","ne","no"],"h":["な","に","ぬ","ね","の"],"k":["ナ","ニ","ヌ","ネ","ノ"]},
 {"head":"は行","r":["ha","hi","fu","he","ho"],"h":["は","ひ","ふ","へ","ほ"],"k":["ハ","ヒ","フ","ヘ","ホ"]},
 {"head":"ま行","r":["ma","mi","mu","me","mo"],"h":["ま","み","む","め","も"],"k":["マ","ミ","ム","メ","モ"]},
 {"head":"や行","r":["ya",B,"yu",B,"yo"],"h":["や",B,"ゆ",B,"よ"],"k":["ヤ",B,"ユ",B,"ヨ"]},
 {"head":"ら行","r":["ra","ri","ru","re","ro"],"h":["ら","り","る","れ","ろ"],"k":["ラ","リ","ル","レ","ロ"]},
 {"head":"わ行","r":["wa",B,B,B,"wo"],"h":["わ",B,B,B,"を"],"k":["ワ",B,B,B,"ヲ"]},
 {"head":"撥音","r":["n",B,B,B,B],"h":["ん",B,B,B,B],"k":["ン",B,B,B,B]},
]
dakuon=[
 {"head":"が行","r":["ga","gi","gu","ge","go"],"h":["が","ぎ","ぐ","げ","ご"],"k":["ガ","ギ","グ","ゲ","ゴ"]},
 {"head":"ざ行","r":["za","ji","zu","ze","zo"],"h":["ざ","じ","ず","ぜ","ぞ"],"k":["ザ","ジ","ズ","ゼ","ゾ"]},
 {"head":"だ行","r":["da","ji","zu","de","do"],"h":["だ","ぢ","づ","で","ど"],"k":["ダ","ヂ","ヅ","デ","ド"]},
 {"head":"ば行","r":["ba","bi","bu","be","bo"],"h":["ば","び","ぶ","べ","ぼ"],"k":["バ","ビ","ブ","ベ","ボ"]},
 {"head":"ぱ行","r":["pa","pi","pu","pe","po"],"h":["ぱ","ぴ","ぷ","ぺ","ぽ"],"k":["パ","ピ","プ","ペ","ポ"]},
]
youon=[
 {"head":"き","r":["kya","kyu","kyo"],"h":["きゃ","きゅ","きょ"],"k":["キャ","キュ","キョ"]},
 {"head":"し","r":["sha","shu","sho"],"h":["しゃ","しゅ","しょ"],"k":["シャ","シュ","ショ"]},
 {"head":"ち","r":["cha","chu","cho"],"h":["ちゃ","ちゅ","ちょ"],"k":["チャ","チュ","チョ"]},
 {"head":"に","r":["nya","nyu","nyo"],"h":["にゃ","にゅ","にょ"],"k":["ニャ","ニュ","ニョ"]},
 {"head":"ひ","r":["hya","hyu","hyo"],"h":["ひゃ","ひゅ","ひょ"],"k":["ヒャ","ヒュ","ヒョ"]},
 {"head":"み","r":["mya","myu","myo"],"h":["みゃ","みゅ","みょ"],"k":["ミャ","ミュ","ミョ"]},
 {"head":"り","r":["rya","ryu","ryo"],"h":["りゃ","りゅ","りょ"],"k":["リャ","リュ","リョ"]},
 {"head":"ぎ","r":["gya","gyu","gyo"],"h":["ぎゃ","ぎゅ","ぎょ"],"k":["ギャ","ギュ","ギョ"]},
 {"head":"じ","r":["ja","ju","jo"],"h":["じゃ","じゅ","じょ"],"k":["ジャ","ジュ","ジョ"]},
 {"head":"び","r":["bya","byu","byo"],"h":["びゃ","びゅ","びょ"],"k":["ビャ","ビュ","ビョ"]},
 {"head":"ぴ","r":["pya","pyu","pyo"],"h":["ぴゃ","ぴゅ","ぴょ"],"k":["ピャ","ピュ","ピョ"]},
]
KH={"あ行":"ア行","か行":"カ行","さ行":"サ行","た行":"タ行","な行":"ナ行","は行":"ハ行","ま行":"マ行",
"や行":"ヤ行","ら行":"ラ行","わ行":"ワ行","が行":"ガ行","ざ行":"ザ行","だ行":"ダ行","ば行":"バ行",
"ぱ行":"パ行","き":"キ","し":"シ","ち":"チ","に":"ニ","ひ":"ヒ","み":"ミ","り":"リ","ぎ":"ギ",
"じ":"ジ","び":"ビ","ぴ":"ピ","撥音":"撥音"}
SMALL="ぁぃぅぇぉゃゅょゎっァィゥェォャュョヮッヵヶ"
def fmt(s): return "".join('<span class="sm">%s</span>'%c if c in SMALL else c for c in s)

V=["a","i","u","e","o"]
zoom=[]   # (id, kana, romaji, altlabel)
zc=[0]

## 3. 五十音表（兩個分頁的表格）
`chart()` 會產生一整頁的表格，同時把每個假名登記到 `zoom` 清單（之後用來產生放大卡）。

In [3]:
def chart(sc):
    out=[]
    other = "k" if sc=="h" else "h"
    othername = "片假名" if sc=="h" else "平假名"
    blocks=[("清音 SEION","基本 46 音",seion,V,""),
            ("濁音・半濁音 DAKUON","加上 ゛ 或 ゜",dakuon,V,""),
            ("拗音 YŌON","小字 ゃ ゅ ょ 拼合",youon,["ya","yu","yo"]," yo")]
    for title,note,cols,vows,cls in blocks:
        out.append('<section class="%s"><h2>%s <span>%s</span></h2><div class="grid">'%(cls.strip(),title,note))
        out.append('<div class="col gutter"><div class="head"></div>%s</div>'%
                   "".join('<div class="vlabel">%s</div>'%v for v in vows))
        for col in cols:
            head = KH.get(col["head"],col["head"]) if sc=="k" else col["head"]
            out.append('<div class="col"><div class="head">%s</div>'%fmt(head))
            for i,kana in enumerate(col[sc]):
                if not kana:
                    out.append('<div class="cell blank"></div>'); continue
                zc[0]+=1; zid="z%d"%zc[0]
                zoom.append((zid,kana,col["r"][i],othername+" "+col[other][i],sc))
                out.append('<label class="cell" for="%s"><span class="kana">%s</span>'
                           '<span class="romaji">%s</span></label>'%(zid,fmt(kana),col["r"][i]))
            out.append('</div>')
        out.append('</div></section>')
    return "".join(out)

## 4. 字卡測驗的牌組

In [4]:
ALL=seion+dakuon
scopes=[]
for i,c in enumerate(ALL):
    scopes.append(("c%d"%i, c["head"], KH.get(c["head"],c["head"])))
for i,v in enumerate(V):
    scopes.append(("r%d"%i, v+" 段", v+" 段"))

import random
fc=[0]
def decks():
    out=['<div class="deck">']
    for sc in ("h","k"):
        items=[]
        for n,col in enumerate(ALL):
            for i,kana in enumerate(col[sc]):
                if kana: items.append((kana,col["r"][i],n,i))
        N=len(items)
        for idx,(kana,rom,n,i) in enumerate(items):
            fc[0]+=1
            style="--i:%d"%idx
            out.append('<input type="checkbox" class="fin" id="f%d">'
                       '<label class="fcard sc-%s k-c%d k-r%d" for="f%d" style="%s"><span class="inner">'
                       '<span class="face fk"><span class="big">%s</span></span>'
                       '<span class="face fr"><span class="rom">%s</span></span>'
                       '</span></label>'%(fc[0],sc,n,i,fc[0],style,fmt(kana),rom))
    out.append('</div>')
    return "".join(out)

deckhtml=decks()

## 5. 讀取單字表與常用短句
這裡讀 `五十音單字表_草稿.xlsx`。片假名的「原文」如果和英文意思重複就不顯示。

In [5]:
import json
import pandas as pd
_df=pd.read_excel(XLSX, sheet_name="單字表")
WORDS={}
for _,row in _df.iterrows():
    if str(row["採用"]).strip()!="是": continue
    w=row["單字"]
    if not isinstance(w,str) or not w.strip(): continue
    sc="h" if row["種類"]=="平假名" else "k"
    try: pos=int(row["目標假名位置"])
    except: pos=0
    _en=row["英文意思"]; _en="" if not isinstance(_en,str) else _en.strip()
    _kj=row["漢字／原文"]; _kj="" if not isinstance(_kj,str) else _kj.strip()
    if sc=="k" and _kj and _kj.lower()==_en.lower(): _kj=""   # 片假名原文與英文重複就不顯示
    WORDS.setdefault((sc,row["假名"].strip()),[]).append(
        (w.strip(), str(row["羅馬拼音"]).strip(), str(row["中文意思"]).strip(), pos,
         len(row["假名"].strip()), _en, _kj))
PH=pd.read_excel(XLSX, sheet_name="常用短句")

## 6. 讀取記憶法
從 `50音記憶_整理版.docx` 的標題階層抓出每個假名的字形與發音聯想。

In [6]:
# ---- 記憶法（來源：50音記憶_整理版.docx）----
from docx import Document as _Doc
MEMO={}
try:
    _d=_Doc(str(DOCX))
    _sc=None; _cur=None
    for _p in _d.paragraphs:
        _t=_p.text.replace("\u3000"," ").strip()
        if not _t: continue
        _pPr=_p._p.pPr
        _st=(_pPr.pStyle.val if (_pPr is not None and _pPr.pStyle is not None) else "") or ""
        _st=_st.replace(" ","")
        if _st=="Heading1":
            _sc = "h" if _t.startswith("平假名") else ("k" if _t.startswith("片假名") else _sc)
            continue
        if _st=="Heading3":
            _cur=_t.split()[0][:1]
            continue
        if _cur and _sc:
            if _t.startswith("字形"):
                MEMO.setdefault((_sc,_cur),{})["shape"]=_t[2:].strip()
            elif _t.startswith("發音"):
                MEMO.setdefault((_sc,_cur),{})["sound"]=_t[2:].strip()
except Exception as _e:
    print("memo skipped:",_e)
print("memo entries:",len(MEMO))

def esc(t):
    return t.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")

def memo_html(sc,kana):
    m=MEMO.get((sc,kana))
    if not m or not (m.get("shape") or m.get("sound")): return ""
    rows=""
    if m.get("shape"): rows+='<span class="mrow"><span class="ml">字形</span><span>%s</span></span>'%esc(m["shape"])
    if m.get("sound"): rows+='<span class="mrow"><span class="ml">發音</span><span>%s</span></span>'%esc(m["sound"])
    return ('<span class="zmemo"><button type="button" class="mtog">💡 記憶法</button>'
            '<span class="mbox">%s</span></span>')%rows

memo entries: 91


## 7. 放大字卡裡的單字區塊

In [7]:
def words_html(sc,kana):
    ws=WORDS.get((sc,kana))
    if not ws: return ""
    out=['<span class="zw">']
    for w,rom,mean,pos,ln,en,kj in ws:
        if pos>0:
            i=pos-1
            disp=w[:i]+'<b>'+w[i:i+ln]+'</b>'+w[i+ln:]
        else: disp=w
        kjh=('<span class="kj">%s</span>'%esc(kj)) if kj else ""
        out.append('<button type="button" class="w" data-w="%s">'
                   '<span class="wtop"><span class="wk">%s</span><span class="wr">%s</span></span>'
                   '<span class="wm">%s<span class="zh">%s</span><span class="en">%s</span></span>'
                   '</button>'%(w,disp,rom,kjh,mean,en))
    out.append('</span>')
    return "".join(out)

## 8. 常用短句的示意圖示（內嵌 SVG）

In [8]:
ICON={
"すみません":'<path d="M15 25v7a10 10 0 0 0 10 10h2a9 9 0 0 0 9-9V23"/><path d="M15 25v-6a2.5 2.5 0 0 1 5 0v3"/><path d="M20 22v-8a2.5 2.5 0 0 1 5 0v8"/><path d="M25 22v-6a2.5 2.5 0 0 1 5.5 0v6"/><path d="M30.5 23v-4a2.75 2.75 0 0 1 5.5 0v4"/>',
"ありがとうございます":'<path d="M24 39s-13-8-13-16a7 7 0 0 1 13-4 7 7 0 0 1 13 4c0 8-13 16-13 16z"/>',
"おねがいします":'<path d="M23 38V13c0-2-2.5-2.5-4-.5-2.5 3-6 8-6 13v4c0 5 4 8.5 10 8.5z"/><path d="M25 38V13c0-2 2.5-2.5 4-.5 2.5 3 6 8 6 13v4c0 5-4 8.5-10 8.5z"/><path d="M17 41h14"/>',
"これをください":'<rect x="26" y="16" width="15" height="15" rx="2"/><path d="M7 24h14"/><path d="M17 19l5 5-5 5"/>',
"いくらですか":'<circle cx="24" cy="24" r="14"/><text x="24" y="30" text-anchor="middle" font-size="17">\u00a5</text>',
"トイレはどこですか":'<rect x="8" y="9" width="32" height="30" rx="4"/><text x="24" y="30" text-anchor="middle" font-size="14">WC</text>',
"メニューをおねがいします":'<rect x="13" y="8" width="22" height="32" rx="3"/><path d="M19 17h10M19 24h10M19 31h7"/>',
"チェックインおねがいします":'<path d="M7 36V15"/><path d="M7 27h34v9"/><path d="M41 27v-4a6 6 0 0 0-6-6H20v10"/><circle cx="14" cy="21" r="3.5"/>',
"にほんごがわかりません":'<path d="M8 11h32v20H26l-8 8v-8H8z"/><text x="24" y="26" text-anchor="middle" font-size="14">?</text><path d="M10 41L39 9"/>',
"えいごがはなせますか":'<path d="M8 11h32v20H26l-8 8v-8H8z"/><text x="24" y="26" text-anchor="middle" font-size="14">A</text>',
"だいじょうぶです":'<circle cx="24" cy="24" r="14"/><path d="M17 24l5 5 9-10"/>',
"おいしいです":'<path d="M8 24h32a16 16 0 0 1-32 0z"/><path d="M6 40h36"/><path d="M18 16c0-3 3-3 3-6M28 16c0-3 3-3 3-6"/>',
"もういちどおねがいします":'<path d="M38 24a14 14 0 1 1-4.5-10.2"/><path d="M38 7v8h-8"/>',
"しゃしんをとってもいいですか":'<rect x="6" y="15" width="36" height="24" rx="4"/><circle cx="24" cy="27" r="7"/><path d="M17 15l3-5h8l3 5"/>',
"たすけてください":'<path d="M24 8l16 30H8L24 8z"/><path d="M24 20v8"/><circle cx="24" cy="33" r="1.6" fill="currentColor" stroke="none"/>',
}
def icon(t):
    d=ICON.get(t)
    if not d: return ""
    return '<svg class="picon" viewBox="0 0 48 48">%s</svg>'%d

## 9. 單字測驗與句型練習的資料
把單字對應到「行／段」的範圍代號，並整理出句型練習用的名詞池。

In [9]:
KANA_POS={}
def _reg(sc,k,keys):
    if k: KANA_POS.setdefault((sc,k),[]).extend(keys)
for _n,_c in enumerate(seion):
    for _i in range(5):
        _reg("h",_c["h"][_i],["c%d"%_n,"r%d"%_i]); _reg("k",_c["k"][_i],["c%d"%_n,"r%d"%_i])
for _n,_c in enumerate(dakuon):
    for _i in range(5):
        _reg("h",_c["h"][_i],["d%d"%_n,"r%d"%_i]); _reg("k",_c["k"][_i],["d%d"%_n,"r%d"%_i])
for _n,_c in enumerate(youon):
    for _i in range(3):
        _reg("h",_c["h"][_i],["y%d"%_n]); _reg("k",_c["k"][_i],["y%d"%_n])

WCARDS={"h":{},"k":{}}
for (sc,kana),lst in WORDS.items():
    keys=KANA_POS.get((sc,kana))
    if not keys: continue
    for w,rom,mean,p,ln,en,kj in lst:
        e=WCARDS[sc].setdefault(w,{"rom":rom,"mean":mean,"en":en,"kj":kj,"sc":set()})
        e["sc"].update(keys)

WSCOPES =[("c%d"%n, c["head"], KH.get(c["head"],c["head"])) for n,c in enumerate(seion)]
WSCOPES+=[("d%d"%n, c["head"], KH.get(c["head"],c["head"])) for n,c in enumerate(dakuon)]
WSCOPES+=[("y%d"%n, c["h"][0]+"行", c["k"][0]+"行") for n,c in enumerate(youon)]
WSCOPES+=[("r%d"%i, v+" 段", v+" 段") for i,v in enumerate(V)]

wfc=[0]
def wdeck_html():
    out=['<div class="wdeck">']
    for sc in ("h","k"):
        items=sorted(WCARDS[sc].items(), key=lambda kv: min(kv[1]["sc"]))
        for idx,(w,e) in enumerate(items):
            wfc[0]+=1
            cls=" ".join("wc-"+x for x in sorted(e["sc"]))
            kj=e["kj"]
            out.append('<input type="checkbox" class="fin" id="wq%d">'
              '<label class="fcard wq wsc-%s %s%s" for="wq%d" style="--i:%d"><span class="inner">'
              '<span class="face fk">'
                '<span class="qw"><span class="wfk">%s</span>%s</span>'
                '<span class="qm"><span class="wmean">%s</span><span class="wen">%s</span></span>'
              '</span>'
              '<span class="face fr">'
                '<span class="wans">%s</span><span class="wrom">%s</span>%s'
                '<span class="wmean">%s</span><span class="wen">%s</span></span>'
              '</span></label>'%(wfc[0],sc,cls," haskj" if kj else "",wfc[0],idx,
                 esc(w), ('<span class="wfj">%s</span>'%esc(kj)) if kj else "",
                 esc(e["mean"]),esc(e["en"]),
                 esc(w),esc(e["rom"]),
                 ('<span class="kj">%s</span>'%esc(kj)) if kj else "",
                 esc(e["mean"]),esc(e["en"])))
    out.append('</div>')
    return "".join(out)

# ---- 句型練習：名詞池（排除形容詞、動詞、招呼語、數量詞、時間詞）----
NOT_NOUN=set("""ありがとう いくら いいです うまい きれい これ すみません たかい やすい のる みる
わかる もういちど みずをください とうきょうへ ろく つぎ るす けさ むかし らいねん キロ ノック
ミス セール""".split())
def spool():
    out={"h":[],"k":[]}
    for sc in ("h","k"):
        for w,e in sorted(WCARDS[sc].items(), key=lambda kv: min(kv[1]["sc"])):
            if w in NOT_NOUN: continue
            out[sc].append({"w":w,"rom":e["rom"],"mean":e["mean"],"en":e["en"],
                            "kj":e["kj"],"sc":sorted(e["sc"])})
    return out

## 10. 常用短句頁

In [10]:
def phrases_html():
    out=['<section class="phrases"><h2>常用短句 PHRASES <span>點一下發音並顯示意思</span></h2><div class="plist">']
    for _,r in PH.iterrows():
        if str(r["採用"]).strip()!="是": continue
        kj=r["漢字寫法"]; kj="" if not isinstance(kj,str) else kj.strip()
        _en=r["英文意思"]; _en="" if not isinstance(_en,str) else _en.strip()
        _t=r["短句（假名）"].strip()
        out.append(('<button type="button" class="p" data-w="%s">'+icon(_t)+'<span class="ptx"><span class="pk">%s</span>'
                   '<span class="pr">%s</span>'
                   '<span class="pm">%s<span class="zh">%s</span><span class="en">%s</span></span></span></button>')
                   %(r["短句（假名）"].strip(), r["短句（假名）"].strip(),
                     str(r["羅馬拼音"]).strip(),
                     ('<span class="kj">%s</span>'%esc(kj)) if kj else "",
                     str(r["中文意思"]).strip(), _en))
    out.append('</div></section>')
    return "".join(out)

## 11. 產生 CSS 規則與隱藏的選項按鈕
分頁切換、範圍複選、翻牌等互動大多是純 CSS，這裡把對應的規則字串組出來。

In [11]:
rules=[]
for key,_,_ in scopes:
    rules.append("#qs-%s:checked ~ .wrap .k-%s{display:block}"%(key,key))
for key,_,_ in WSCOPES:
    rules.append("#ws-%s:checked ~ .wrap .wc-%s{display:block}"%(key,key))
rules.append("#wk-h:checked ~ .wrap .wsc-k{display:none}")
rules.append("#wk-k:checked ~ .wrap .wsc-h{display:none}")
for key,_,_ in WSCOPES:
    rules.append("#ws-%s:checked ~ .wrap label[for=ws-%s]{background:var(--accent);border-color:var(--accent);color:#fff}"%(key,key))
for key,_,_ in WSCOPES:
    rules.append("#ss-%s:checked ~ .wrap label[for=ss-%s]{background:var(--accent);border-color:var(--accent);color:#fff}"%(key,key))
rules.append("#qk-h:checked ~ .wrap .sc-k{display:none}")
rules.append("#qk-k:checked ~ .wrap .sc-h{display:none}")
for key,_,_ in scopes:
    rules.append("#qs-%s:checked ~ .wrap label[for=qs-%s]{background:var(--accent);border-color:var(--accent);color:#fff}"%(key,key))
GEN="\n".join(rules)

radios=[]
radios.append('<input type="radio" name="tab" id="t-h" class="hid" checked>')
radios.append('<input type="radio" name="tab" id="t-k" class="hid">')
radios.append('<input type="radio" name="tab" id="t-q" class="hid">')
radios.append('<input type="radio" name="tab" id="t-p" class="hid">')
radios.append('<input type="radio" name="tab" id="t-w" class="hid">')
radios.append('<input type="radio" name="tab" id="t-s" class="hid">')
radios.append('<input type="radio" name="tab" id="t-n" class="hid">')
for i in range(1,6):
    radios.append('<input type="checkbox" id="nr-%d" class="hid"%s>'%(i," checked" if i==1 else ""))
radios.append('<input type="radio" name="nf" id="nf-n" class="hid" checked>')
radios.append('<input type="radio" name="nf" id="nf-k" class="hid">')
radios.append('<input type="radio" name="sk" id="sk-h" class="hid" checked>')
radios.append('<input type="radio" name="sk" id="sk-k" class="hid">')
for n,(key,_,_) in enumerate(WSCOPES):
    radios.append('<input type="checkbox" id="ss-%s" class="hid"%s>'%(key," checked" if n==1 else ""))
radios.append('<input type="radio" name="sm" id="sm-a" class="hid" checked>')
radios.append('<input type="radio" name="sm" id="sm-j" class="hid">')
radios.append('<input type="radio" name="qk" id="qk-h" class="hid" checked>')
radios.append('<input type="radio" name="qk" id="qk-k" class="hid">')
for n,(key,_,_) in enumerate(scopes):
    radios.append('<input type="checkbox" id="qs-%s" class="hid"%s>'%(key," checked" if n==1 else ""))
radios.append('<input type="radio" name="wk" id="wk-h" class="hid" checked>')
radios.append('<input type="radio" name="wk" id="wk-k" class="hid">')
for n,(key,_,_) in enumerate(WSCOPES):
    radios.append('<input type="checkbox" id="ws-%s" class="hid"%s>'%(key," checked" if n==1 else ""))
radios.append('<input type="radio" name="wf" id="wf-k" class="hid" checked>')
radios.append('<input type="radio" name="wf" id="wf-r" class="hid">')
for n in ("n","v"):
    radios.append('<input type="radio" name="wo" id="wo-%s" class="hid"%s>'%(n," checked" if n=="n" else ""))
radios.append('<input type="radio" name="wm" id="wm-a" class="hid" checked>')
radios.append('<input type="radio" name="wm" id="wm-j" class="hid">')
radios.append('<input type="radio" name="qf" id="qf-k" class="hid" checked>')
radios.append('<input type="radio" name="qf" id="qf-r" class="hid">')
for n in ("n","v"):
    radios.append('<input type="radio" name="qo" id="qo-%s" class="hid"%s>'%(n," checked" if n=="n" else ""))

def _wchips(pref):
    return "".join('<label for="ws-%s"><span class="lh">%s</span><span class="lk">%s</span></label>'%(k,fmt(a),fmt(b))
                   for k,a,b in WSCOPES if k[0]==pref)
wchips_col=_wchips("c"); wchips_dak=_wchips("d"); wchips_yo=_wchips("y")
wchips_row="".join('<label for="ws-%s">%s</label>'%(k,a) for k,a,b in WSCOPES if k[0]=="r")

chips_col="".join('<label for="qs-%s"><span class="lh">%s</span><span class="lk">%s</span></label>'%(k,fmt(a),fmt(b))
                  for k,a,b in scopes if k[0]=="c")
chips_row="".join('<label for="qs-%s">%s</label>'%(k,a) for k,a,b in scopes if k[0]=="r")

## 12. 組出各頁內容

In [12]:
import pandas as pd
CH_H=chart("h")
CH_K=chart("k")
zoomhtml="".join('<input type="checkbox" class="zin" id="%s"><div class="zview" data-k="%s">'
  '<label class="zbg" for="%s"></label>'
  '<span class="zsp"><span class="zk">%s</span><span class="zsvg"></span></span>'
  '<span class="zr">%s</span><span class="za">%s</span>%s%s'
  '<span class="zbtns"><button type="button" class="b-stk">\U0001F501 \u7b46\u9806\u91cd\u64ad</button>'
  '<label class="zclose" for="%s">\u95dc\u9589 \u2715</label></span></div>'
  %(zid,k,zid,fmt(k),r,fmt(a),memo_html(sc,k),words_html(sc,k),zid) for zid,k,r,a,sc in zoom)

KVGJSON=KVG_PATH.read_text(encoding="utf-8")

## 13. HTML 模板
整份網頁的骨架與樣式，佔位符會在下一步被替換掉。

In [13]:
HTML = """<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>日文五十音表</title>
<style>
:root{--bg:#faf7f2;--card:#fff;--ink:#1f2430;--muted:#8a8f9c;--line:#e7e2d9;
  --accent:#3f6fd8;--accent-soft:#eaf0fd;}
*{box-sizing:border-box;-webkit-tap-highlight-color:transparent;}
html,body{margin:0;padding:0;}
body{background:var(--bg);color:var(--ink);
  font-family:"Hiragino Sans","Hiragino Kaku Gothic ProN","Noto Sans JP","PingFang TC",-apple-system,sans-serif;
  padding:16px 18px 60px;}
.hid{position:fixed;top:-200px;left:-200px;opacity:0;width:1px;height:1px;}
header{display:flex;align-items:center;justify-content:space-between;gap:16px;flex-wrap:wrap;margin-bottom:14px;}
h1{font-size:clamp(18px,2.2vw,26px);margin:0;letter-spacing:.04em;font-weight:700;}
h1 small{display:block;font-size:.55em;color:var(--muted);font-weight:400;letter-spacing:.08em;margin-top:4px;}
.tabs{display:flex;gap:8px;background:#efeae1;padding:5px;border-radius:14px;}
.tabs label{font-size:clamp(14px,1.7vw,19px);font-weight:600;color:var(--muted);
  padding:10px 22px;border-radius:10px;cursor:pointer;transition:.2s;}
.hint{font-size:13px;color:var(--muted);margin:0 0 18px;}
.page{display:none;}
section{margin-bottom:30px;}
h2{font-size:clamp(14px,1.5vw,17px);margin:0 0 10px;font-weight:700;color:var(--accent);
  letter-spacing:.1em;display:flex;align-items:center;gap:12px;}
h2::after{content:"";flex:1;height:1px;background:var(--line);}
h2 span{color:var(--muted);font-weight:400;font-size:.85em;letter-spacing:.02em;}
.grid{display:flex;gap:clamp(4px,.7vw,10px);align-items:stretch;}
.col{flex:1 1 0;min-width:0;display:flex;flex-direction:column;gap:clamp(4px,.7vw,10px);}
.col.gutter{flex:0 0 clamp(24px,3vw,40px);}
.head{text-align:center;font-size:clamp(10px,1.05vw,13px);color:var(--muted);
  letter-spacing:.06em;height:20px;line-height:20px;}
.cell{background:var(--card);border:1px solid var(--line);border-radius:12px;display:flex;
  flex-direction:column;align-items:center;justify-content:center;
  min-height:clamp(56px,8.2vw,96px);gap:2px;cursor:pointer;padding:4px 2px;
  transition:transform .12s,box-shadow .12s,border-color .12s;}
.cell:active{transform:scale(.94);border-color:var(--accent);box-shadow:0 0 0 2px var(--accent-soft);}
.cell .kana{font-size:clamp(22px,3.4vw,42px);line-height:1.05;font-weight:500;}
.cell .romaji{font-size:clamp(9px,1.15vw,15px);color:var(--muted);letter-spacing:.06em;
  font-family:-apple-system,"Helvetica Neue",sans-serif;}
.cell.blank{background:transparent;border:1px dashed #e9e4db;cursor:default;}
.sm{font-size:.62em;vertical-align:-.06em;}
.vlabel{display:flex;align-items:center;justify-content:center;min-height:clamp(56px,8.2vw,96px);
  color:var(--muted);font-size:clamp(11px,1.3vw,16px);font-family:-apple-system,sans-serif;}
/* 放大卡 */
.zview{position:fixed;inset:0;display:none;flex-direction:column;align-items:center;justify-content:center;
  background:#faf7f2;z-index:60;}
.zin:checked + .zview{display:flex;}
.zk{font-size:min(30vh,28vw);line-height:1;font-weight:500;}
.zr{font-size:min(9vh,9vw);color:var(--accent);letter-spacing:.14em;margin-top:2vh;font-weight:600;
  font-family:-apple-system,"Helvetica Neue",sans-serif;}
.za{font-size:min(4vh,4vw);color:var(--muted);margin-top:2.5vh;letter-spacing:.1em;}
.zin{position:fixed;top:-200px;left:-200px;opacity:0;width:1px;height:1px;}
.zbg{position:absolute;inset:0;cursor:pointer;}
.zview > span{position:relative;z-index:1;}
.zsvg{display:none;}
.zsvg{align-items:flex-end;}

.zsvg svg.smsvg{width:min(19vh,18vw);height:min(19vh,18vw);}
.zsvg svg path{fill:none;stroke-width:4.5;stroke-linecap:round;stroke-linejoin:round;}
.zsvg svg path.gh{stroke:#e6e2d9;}
.zsvg svg path.ln{stroke:var(--ink);}
.zview.stroke .zk{display:none;}
.zview.stroke .zsvg{display:flex;gap:2vw;}
.zsp{display:flex;align-items:center;justify-content:center;cursor:pointer;}
body:not(.js) .zsp{cursor:default;}
.plist{display:flex;flex-wrap:wrap;gap:8px;}
.picon{width:clamp(34px,4vw,46px);height:clamp(34px,4vw,46px);flex:0 0 auto;
  fill:none;stroke:var(--accent);stroke-width:2;stroke-linecap:round;stroke-linejoin:round;}
.picon text{stroke:none;fill:var(--accent);font-family:-apple-system,"Helvetica Neue",sans-serif;font-weight:700;}
.ptx{display:flex;flex-direction:column;gap:3px;min-width:0;}
.p{flex:1 1 320px;display:flex;flex-direction:row;align-items:center;gap:14px;text-align:left;
  background:var(--card);border:1px solid var(--line);border-radius:12px;padding:10px 14px;
  font-family:inherit;color:var(--ink);cursor:pointer;}
.p:active{background:var(--accent-soft);}
.pk{font-size:clamp(16px,1.9vw,21px);font-weight:500;}
.pm{display:none;font-size:clamp(12px,1.4vw,15px);color:var(--muted);gap:8px;align-items:center;}
.pm .zh{color:var(--ink);}
.p.show .pm{display:flex;flex-wrap:wrap;}
.pm .zh::after{content:"・";margin-left:8px;opacity:.45;}
.pr{font-size:clamp(11px,1.25vw,14px);color:var(--accent);font-weight:600;letter-spacing:.04em;
  font-family:-apple-system,"Helvetica Neue",sans-serif;}
body:not(.js) .p{cursor:default;}
.zview{overflow-y:auto;padding:3vh 5vw;}
.zmemo{display:flex;flex-direction:column;align-items:center;gap:8px;margin-top:1.8vh;}
.mtog{border:1px solid var(--line);background:var(--card);color:var(--accent);font-family:inherit;
  font-size:clamp(13px,1.5vw,16px);font-weight:600;padding:8px 18px;border-radius:20px;cursor:pointer;}
.zmemo.show .mtog{background:var(--accent);border-color:var(--accent);color:#fff;}
.mbox{display:none;max-width:min(620px,88vw);background:var(--accent-soft);border-radius:14px;
  padding:12px 16px;gap:6px;text-align:left;}
.zmemo.show .mbox{display:flex;flex-direction:column;}
.mrow{display:flex;gap:10px;font-size:clamp(13px,1.5vw,16px);line-height:1.6;}
.ml{flex:0 0 auto;color:var(--muted);font-size:.9em;padding-top:.15em;}
body:not(.js) .zmemo{display:none;}
.zw{display:flex;flex-direction:column;gap:6px;margin-top:1.6vh;align-items:center;}
.zw .w{display:flex;flex-direction:column;align-items:center;gap:3px;background:var(--card);
  border:1px solid var(--line);border-radius:12px;padding:8px 16px;cursor:pointer;
  font-family:inherit;color:var(--ink);}
.zw .w:active{background:var(--accent-soft);}
.zw .wk{font-size:clamp(19px,2.4vw,26px);font-weight:500;}
.zw .wk b{color:var(--accent);font-weight:700;}
.zw .wr{font-size:clamp(12px,1.4vw,15px);color:var(--accent);font-weight:600;letter-spacing:.05em;
  font-family:-apple-system,"Helvetica Neue",sans-serif;}
/* ===== 統一資訊層級 =====
   日文本體：深色最大　拼音：主題色　漢字/原文：灰框小字　中文：深色　英文：灰色 */
.kj{border:1px solid var(--line);border-radius:6px;padding:1px 7px;color:var(--muted);
  font-size:.92em;line-height:1.5;white-space:nowrap;}
.wtop{display:flex;align-items:baseline;gap:10px;}
.zw .wm{display:none;font-size:clamp(12px,1.4vw,15px);color:var(--muted);gap:8px;align-items:center;}
.zw .wm .zh{color:var(--ink);}
.zw .w.show .wm{display:flex;}
.wm .en{opacity:.8;}
.wm .zh::after{content:"・";margin-left:8px;opacity:.45;}
.zsvg svg{width:min(30vh,28vw);height:min(30vh,28vw);}
.zbtns{position:relative;z-index:2;margin-top:4.5vh;display:flex;gap:10px;align-items:center;flex-wrap:wrap;justify-content:center;}
.zbtns button,.zclose{border:1px solid var(--line);background:var(--card);color:var(--accent);
  font-family:inherit;font-size:clamp(13px,1.6vw,17px);font-weight:600;padding:10px 20px;
  border-radius:12px;cursor:pointer;}
.zclose{color:var(--muted);}
body:not(.js) .zbtns button{display:none;}
.jsonly{display:none;}
body.js .jsonly{display:inline-flex;}
body.js p.jsonly{display:block;}
.credit{margin-top:34px;font-size:11px;color:#b9b3a8;letter-spacing:.04em;line-height:1.7;}
.credit a{color:#b9b3a8;}
/* 測驗 */
.panel{background:var(--card);border:1px solid var(--line);border-radius:16px;padding:14px 16px;
  margin-bottom:22px;display:flex;flex-direction:column;gap:12px;}
.opt{display:flex;align-items:flex-start;gap:12px;flex-wrap:wrap;}
.opt > b{font-size:clamp(11px,1.15vw,14px);font-weight:600;color:var(--muted);letter-spacing:.1em;
  min-width:5.2em;line-height:2.6;}
.seg{display:flex;gap:6px;background:#f2efe9;padding:4px;border-radius:11px;flex-wrap:wrap;}
.seg label{color:var(--muted);font-size:clamp(13px,1.4vw,16px);font-weight:600;padding:8px 16px;
  border-radius:8px;cursor:pointer;transition:.15s;}
.chips{display:flex;gap:6px;flex-wrap:wrap;flex:1;}
.chips label{border:1px solid var(--line);background:var(--card);color:var(--ink);
  font-size:clamp(13px,1.4vw,16px);padding:8px 14px;border-radius:20px;cursor:pointer;transition:.15s;}
.lk{display:none;}
.tools{display:flex;gap:8px;flex-wrap:wrap;}
.sbtns button{padding:11px 22px;font-size:clamp(14px,1.6vw,17px);}
.tools button{border:1px solid var(--line);background:var(--card);font-family:inherit;
  color:var(--accent);font-size:clamp(13px,1.4vw,16px);font-weight:600;padding:9px 18px;
  border-radius:10px;cursor:pointer;}
.tools button:active{background:var(--accent-soft);}
.deck,.wdeck{display:flex;flex-wrap:wrap;gap:clamp(8px,1.1vw,14px);}
.wq{flex:1 1 clamp(150px,20%,230px);}
.wq{height:clamp(180px,30vh,270px);}
.wq .face,.scard .face{flex-direction:column;gap:5px;padding:10px;}
.wq .face.fr .wmean{font-size:clamp(13px,1.7vw,18px);}
.wq .face.fr .wen{font-size:clamp(10px,1.3vw,14px);}
.wbig{font-size:clamp(22px,3.2vw,38px);font-weight:500;line-height:1.2;text-align:center;}
.wrom{font-size:clamp(15px,2vw,22px);color:var(--accent);font-weight:600;letter-spacing:.05em;
  font-family:-apple-system,"Helvetica Neue",sans-serif;}
.wmean{font-size:clamp(14px,1.8vw,20px);color:var(--ink);}
.wen{font-size:clamp(11px,1.4vw,15px);color:var(--muted);}
/* 數字頁 */
.ngrid{display:flex;flex-wrap:wrap;gap:clamp(6px,.9vw,12px);margin-bottom:14px;}
.ncard{flex:1 1 clamp(110px,13%,160px);background:var(--card);border:1px solid var(--line);
  border-radius:14px;padding:12px 10px;text-align:center;}
.ngrid.wide .ncard{flex:1 1 clamp(150px,20%,240px);}
.nnum{font-size:clamp(20px,2.6vw,30px);font-weight:700;color:var(--ink);}
.nkana{font-size:clamp(15px,1.9vw,21px);margin-top:4px;}
.nrom{font-size:clamp(11px,1.3vw,14px);color:var(--accent);font-weight:600;letter-spacing:.04em;
  font-family:-apple-system,"Helvetica Neue",sans-serif;margin-top:2px;}
.nkj{font-size:clamp(11px,1.3vw,14px);color:var(--muted);margin-top:4px;}
.ncard2{width:clamp(200px,32vw,340px);height:clamp(160px,24vh,210px);perspective:1000px;cursor:pointer;}
.ncard2 .inner{position:relative;display:block;width:100%;height:100%;transform-style:preserve-3d;
  transition:transform .45s cubic-bezier(.4,.2,.2,1);}
.ncard2.flip .inner{transform:rotateY(180deg);}
.ncard2 .face{flex-direction:column;gap:6px;padding:12px;}
.nbig{font-size:clamp(34px,5vw,58px);font-weight:700;}
.nans{font-size:clamp(20px,2.8vw,32px);font-weight:500;}
/* 句型頁 */
.pat{background:var(--card);border:1px solid var(--line);border-radius:16px;
  padding:22px 20px;text-align:center;margin-bottom:16px;}
.pjp{font-size:clamp(24px,3.4vw,40px);font-weight:500;line-height:1.4;}
.blank{color:var(--accent);border-bottom:3px solid var(--accent);padding:0 4px;}
ruby rt{font-size:.45em;color:var(--muted);font-weight:400;}
.gram ruby rt{font-size:.55em;}
.prom{font-size:clamp(14px,1.7vw,20px);color:var(--accent);font-weight:600;letter-spacing:.05em;
  margin-top:8px;font-family:-apple-system,"Helvetica Neue",sans-serif;}
.pzh{font-size:clamp(14px,1.7vw,19px);color:var(--ink);margin-top:10px;}
.gram{background:var(--accent-soft);border-radius:14px;padding:14px 18px;}
.gram p{margin:0 0 8px;font-size:clamp(13px,1.5vw,16px);line-height:1.85;}
.gram p:last-child{margin-bottom:0;}
.gram b{color:var(--accent);}
.sline{display:flex;align-items:center;justify-content:center;gap:clamp(10px,2vw,20px);
  flex-wrap:wrap;margin:18px 0 6px;}
.stail{display:flex;flex-direction:column;gap:4px;}
.sjp{font-size:clamp(22px,3vw,34px);font-weight:500;}
.srom{font-size:clamp(12px,1.4vw,16px);color:var(--accent);font-weight:600;letter-spacing:.04em;
  font-family:-apple-system,"Helvetica Neue",sans-serif;}
.szh{text-align:center;font-size:clamp(14px,1.7vw,19px);color:var(--ink);margin-bottom:14px;}
#szhw{color:var(--accent);font-weight:600;}
.sbtns{justify-content:center;}
.scard{width:clamp(150px,22vw,220px);height:clamp(150px,22vh,200px);perspective:1000px;cursor:pointer;}
.scard .inner{position:relative;display:block;width:100%;height:100%;transform-style:preserve-3d;
  transition:transform .45s cubic-bezier(.4,.2,.2,1);}
.scard.flip .inner{transform:rotateY(180deg);}
.sempty{color:var(--muted);font-size:14px;padding:20px;}
.qw,.qm{display:flex;flex-direction:column;align-items:center;gap:6px;}
.qm{display:none;}
#wf-r:checked ~ .wrap .qw{display:none;}
#wf-r:checked ~ .wrap .qm{display:flex;}
.wfj{display:none;}
#wm-j:checked ~ .wrap .haskj .wfk{display:none;}
#wm-j:checked ~ .wrap .haskj .wfj{display:block;}
.wfk,.wfj{font-size:clamp(22px,3.2vw,38px);font-weight:500;line-height:1.2;text-align:center;}
.wans{font-size:clamp(18px,2.4vw,26px);font-weight:500;}

.fin{position:fixed;top:-200px;opacity:0;width:1px;height:1px;}
.fcard{flex:1 1 clamp(120px,15%,190px);min-width:110px;height:clamp(150px,26vh,240px);
  perspective:1000px;cursor:pointer;display:none;order:0;}
.inner{position:relative;display:block;width:100%;height:100%;
  transition:transform .45s cubic-bezier(.4,.2,.2,1);transform-style:preserve-3d;}
.face{position:absolute;inset:0;backface-visibility:hidden;-webkit-backface-visibility:hidden;
  border-radius:16px;display:flex;align-items:center;justify-content:center;border:1px solid var(--line);
  background:var(--card);box-shadow:0 2px 8px rgba(0,0,0,.05);}
.face.fr{transform:rotateY(180deg);background:var(--accent-soft);border-color:var(--accent);}
.big{font-size:clamp(38px,7vw,80px);line-height:1;font-weight:500;}
.rom{font-size:clamp(26px,4.6vw,54px);color:var(--accent);letter-spacing:.08em;font-weight:600;
  font-family:-apple-system,"Helvetica Neue",sans-serif;}
.yo .cell .kana{font-size:clamp(16px,2.6vw,32px);white-space:nowrap;}
@media (max-width:820px){
  body{padding:12px 10px 40px;}
  header{gap:10px;margin-bottom:10px;}
  h1{font-size:17px;} h1 small{font-size:.5em;}
  .tabs{padding:4px;gap:3px;} .tabs label{padding:8px 9px;font-size:13px;}
  .tabs .jp{display:none;}
  .grid{gap:3px;} .col{gap:3px;} .col.gutter{flex:0 0 16px;}
  .head{font-size:9px;height:16px;line-height:16px;}
  .cell{min-height:clamp(42px,11.5vw,80px);border-radius:9px;padding:2px 0;}
  .cell .kana{font-size:clamp(17px,5vw,34px);}
  .cell .romaji{font-size:clamp(8px,2.3vw,13px);letter-spacing:0;}
  .yo .cell .kana{font-size:clamp(13px,3.6vw,24px);}
  .vlabel{font-size:10px;min-height:clamp(42px,11.5vw,80px);}
  h2{font-size:12px;letter-spacing:.04em;} h2 span{display:none;}
  section{margin-bottom:20px;}
  .panel{padding:10px 12px;gap:9px;border-radius:12px;}
  .opt{gap:6px;} .opt > b{min-width:100%;line-height:1.6;font-size:11px;}
  .seg label,.chips label{padding:7px 12px;font-size:13px;}
  .tools button{padding:8px 12px;font-size:13px;}
  .fcard{flex:1 1 30%;min-width:88px;height:clamp(110px,20vh,170px);}
  .big{font-size:clamp(30px,9vw,56px);} .rom{font-size:clamp(20px,6vw,38px);}
  .zbtns button,.zclose{padding:9px 14px;font-size:14px;}
  .zsvg svg{width:min(24vh,45vw);height:min(24vh,45vw);}
  .zsvg svg.smsvg{width:min(15vh,28vw);height:min(15vh,28vw);}
  .zk{font-size:min(26vh,42vw);}
  .zw{gap:5px;margin-top:1.6vh;}
  .zw .w{padding:7px 12px;gap:8px;}
  .zw .wk{font-size:17px;} .zw .wr,.zw .wm{font-size:12px;}
  .p{flex:1 1 260px;padding:8px 12px;gap:10px;}
  .pat{padding:16px 12px;} .gram{padding:12px 14px;}
  .scard{width:clamp(130px,42vw,190px);height:clamp(130px,20vh,180px);} .pk{font-size:16px;} .pm{font-size:12px;} .pr{font-size:11px;}
  .zr{font-size:min(7vh,11vw);} .za{font-size:min(3.4vh,5vw);}
  .credit{font-size:10px;}
}
@media (max-width:420px){
  body{padding:10px 6px 32px;}
  .grid{gap:2px;} .col{gap:2px;} .col.gutter{flex:0 0 13px;}
  .cell{border-radius:7px;}
  .cell .romaji{font-size:8px;}
  .fcard{flex:1 1 44%;}
  .p{flex:1 1 100%;}
}
/* 分頁切換 */
#t-h:checked ~ .wrap #page-h{display:block}
#t-k:checked ~ .wrap #page-k{display:block}
#t-q:checked ~ .wrap #page-q{display:block}
#t-p:checked ~ .wrap #page-p{display:block}
#t-w:checked ~ .wrap #page-w{display:block}
#t-s:checked ~ .wrap #page-s{display:block}
#t-n:checked ~ .wrap #page-n{display:block}
#t-h:checked ~ .wrap label[for=t-h],
#t-k:checked ~ .wrap label[for=t-k],
#t-q:checked ~ .wrap label[for=t-q],
#t-p:checked ~ .wrap label[for=t-p],
#t-w:checked ~ .wrap label[for=t-w],
#t-s:checked ~ .wrap label[for=t-s],
#t-n:checked ~ .wrap label[for=t-n]{background:var(--card);color:var(--accent);box-shadow:0 1px 4px rgba(0,0,0,.10)}
#t-q:checked ~ .wrap #hint-q{display:block}
#t-q:checked ~ .wrap #hint-w{display:none}
#t-p:checked ~ .wrap #hint-w{display:none}
#t-p:checked ~ .wrap #hint-q{display:none}
#t-p:checked ~ .wrap{--accent:#2f8f6f;--accent-soft:#e8f4ef}
#t-w:checked ~ .wrap{--accent:#b8562f;--accent-soft:#fbeee7}
#t-s:checked ~ .wrap{--accent:#8a5bb8;--accent-soft:#f2ecf9}
#t-n:checked ~ .wrap{--accent:#2f7f8f;--accent-soft:#e6f2f4}
#t-n:checked ~ .wrap #hint-w{display:none}
#t-n:checked ~ .wrap #hint-q{display:none}
#t-n:checked ~ .wrap #hint-x{display:none}
#t-s:checked ~ .wrap #hint-w{display:none}
#t-s:checked ~ .wrap #hint-q{display:none}
#t-s:checked ~ .wrap #hint-x{display:none}
#t-w:checked ~ .wrap #hint-w{display:none}
#t-w:checked ~ .wrap #hint-x{display:block}
#t-w:checked ~ .wrap #hint-q{display:none}
.hint{display:none}
#hint-x{display:none}
#t-k:checked ~ .wrap{--accent:#d9722f;--accent-soft:#fdf0e6}
#t-q:checked ~ .wrap{--accent:#6b4fbb;--accent-soft:#f0ecfb}
#t-k:checked ~ .zin:checked + .zview{--accent:#d9722f}
/* 測驗選項狀態 */
#qk-h:checked ~ .wrap label[for=qk-h],
#qk-k:checked ~ .wrap label[for=qk-k],
#qf-k:checked ~ .wrap label[for=qf-k],
#qf-r:checked ~ .wrap label[for=qf-r],
#wk-h:checked ~ .wrap label[for=wk-h],
#wk-k:checked ~ .wrap label[for=wk-k],
#wf-k:checked ~ .wrap label[for=wf-k],
#wf-r:checked ~ .wrap label[for=wf-r],
#nr-1:checked ~ .wrap label[for=nr-1]{background:var(--accent);border-color:var(--accent);color:#fff}
#nr-2:checked ~ .wrap label[for=nr-2]{background:var(--accent);border-color:var(--accent);color:#fff}
#nr-3:checked ~ .wrap label[for=nr-3]{background:var(--accent);border-color:var(--accent);color:#fff}
#nr-4:checked ~ .wrap label[for=nr-4]{background:var(--accent);border-color:var(--accent);color:#fff}
#nr-5:checked ~ .wrap label[for=nr-5]{background:var(--accent);border-color:var(--accent);color:#fff}
#nf-n:checked ~ .wrap label[for=nf-n],
#nf-k:checked ~ .wrap label[for=nf-k],
#sm-a:checked ~ .wrap label[for=sm-a],
#sm-j:checked ~ .wrap label[for=sm-j],
#sk-h:checked ~ .wrap label[for=sk-h],
#sk-k:checked ~ .wrap label[for=sk-k],
#wm-a:checked ~ .wrap label[for=wm-a],
#wm-j:checked ~ .wrap label[for=wm-j],
#wo-n:checked ~ .wrap label[for=wo-n],
#wo-v:checked ~ .wrap label[for=wo-v],
#qo-n:checked ~ .wrap label[for=qo-n],
#qo-v:checked ~ .wrap label[for=qo-v]{background:var(--card);color:var(--accent);box-shadow:0 1px 3px rgba(0,0,0,.10)}
#qk-k:checked ~ .wrap .lk{display:inline}
#qk-k:checked ~ .wrap .lh{display:none}
#wk-k:checked ~ .wrap #page-w .lk{display:inline}
#wk-k:checked ~ .wrap #page-w .lh{display:none}
#wk-h:checked ~ .wrap #page-w .lk{display:none}
#wk-h:checked ~ .wrap #page-w .lh{display:inline}
/* 卡片順序 */
#qo-v:checked ~ .wrap .deck .fcard{order:calc(0 - var(--i))}
#wo-v:checked ~ .wrap .wq{order:calc(0 - var(--i))}
/* 翻面：預設假名在前 */
.fin:checked + .fcard .inner{transform:rotateY(180deg)}
#qf-r:checked ~ .wrap .deck .fcard .inner{transform:rotateY(180deg)}
#qf-r:checked ~ .wrap .deck .fin:checked + .fcard .inner{transform:rotateY(360deg)}

__GEN__
</style>
</head>
<body>
__RADIOS__
<div class="wrap">
<header>
  <h1>日本語 五十音<small>HIRAGANA ・ KATAKANA</small></h1>
  <div class="tabs">
    <label for="t-h">平假名<span class="jp"> ひらがな</span></label>
    <label for="t-k">片假名<span class="jp"> カタカナ</span></label>
    <label for="t-q">字卡測驗</label>
    <label for="t-w">單字測驗</label>
    <label for="t-s">句型</label>
    <label for="t-n">數字</label>
    <label for="t-p">常用短句</label>
  </div>
</header>
<p class="hint" id="hint-q">行與段可以複選（再點一次取消）；點字卡翻面對答案。</p>
<p class="hint" id="hint-x">行與段可以複選；點卡片翻面看拼音和中英文意思。</p>
<p class="hint jsonly" id="hint-w">卡片裡點任何東西都會唸出來：點大字唸假名，點單字唸單字並顯示中英文意思。</p>

<div class="page" id="page-h">__CHART_H__</div>
<div class="page" id="page-k">__CHART_K__</div>

<div class="page" id="page-s">
  <section>
    <h2>句型 PATTERN <span>喜歡什麼</span></h2>
    <div class="pat">
      <div class="pjp"><span class="blank">＿＿＿</span> が <ruby>好<rt>す</rt></ruby>きです。</div>
      <div class="prom">___ ga suki desu.</div>
      <div class="pzh">我喜歡＿＿＿。</div>
    </div>
    <div class="gram">
      <p><b>が</b> 是助詞，放在喜歡的東西後面，標示出「喜歡的對象」。中文說「我喜歡貓」，日文的語序是「貓　が　喜歡」，對象先講、感受後講。</p>
      <p><b>好<ruby><rt>す</rt></ruby>き（すき）</b> 是「喜歡」，<b>です</b> 讓句子變禮貌。連起來就是「（我）喜歡＿＿＿」。主詞「我」在日文裡通常省略，講話的對象自然知道是你。</p>
      <p>想說不喜歡，把 です 換成 <b>じゃないです</b>：「ねこ が <ruby>好<rt>す</rt></ruby>きじゃないです。」問對方，句尾加 <b>か</b>：「ねこ が <ruby>好<rt>す</rt></ruby>きですか。」</p>
    </div>
  </section>
  <section>
    <h2>練習 PRACTICE <span>點字卡翻面，按下一個換字</span></h2>
    <div class="panel">
      <div class="opt"><b>假名</b><div class="seg">
        <label for="sk-h">平假名 ひらがな</label><label for="sk-k">片假名 カタカナ</label></div></div>
      <div class="opt"><b>清音行</b><div class="chips">__SCHIPS_COL__</div></div>
      <div class="opt"><b>濁音行</b><div class="chips">__SCHIPS_DAK__</div></div>
      <div class="opt"><b>拗音行</b><div class="chips">__SCHIPS_YO__</div></div>
      <div class="opt"><b>橫的段</b><div class="chips">__SCHIPS_ROW__</div></div>
      <div class="opt"><b>出題字形</b><div class="seg">
        <label for="sm-a">全部假名</label><label for="sm-j">有漢字用漢字</label></div></div>
    </div>
    <div class="sline"><div id="sslot"></div>
      <div class="stail"><span class="sjp">が <ruby>好<rt>す</rt></ruby>きです。</span><span class="srom">ga suki desu.</span></div>
    </div>
    <div class="szh">我喜歡 <span id="szhw">＿＿＿</span>。</div>
    <div class="tools sbtns">
      <button type="button" id="btn-ssay">🔊 唸這句</button>
      <button type="button" id="btn-snext">下一個 →</button>
    </div>
  </section>
</div>

<div class="page" id="page-n">
  <section>
    <h2>0 – 10 <span>先把這十一個背熟</span></h2>
    <div class="ngrid"><div class="ncard"><div class="nnum">0</div><div class="nkana">ゼロ／れい</div><div class="nrom">zero / rei</div><div class="nkj">零</div></div><div class="ncard"><div class="nnum">1</div><div class="nkana">いち</div><div class="nrom">ichi</div><div class="nkj">一</div></div><div class="ncard"><div class="nnum">2</div><div class="nkana">に</div><div class="nrom">ni</div><div class="nkj">二</div></div><div class="ncard"><div class="nnum">3</div><div class="nkana">さん</div><div class="nrom">san</div><div class="nkj">三</div></div><div class="ncard"><div class="nnum">4</div><div class="nkana">よん／し</div><div class="nrom">yon / shi</div><div class="nkj">四</div></div><div class="ncard"><div class="nnum">5</div><div class="nkana">ご</div><div class="nrom">go</div><div class="nkj">五</div></div><div class="ncard"><div class="nnum">6</div><div class="nkana">ろく</div><div class="nrom">roku</div><div class="nkj">六</div></div><div class="ncard"><div class="nnum">7</div><div class="nkana">なな／しち</div><div class="nrom">nana / shichi</div><div class="nkj">七</div></div><div class="ncard"><div class="nnum">8</div><div class="nkana">はち</div><div class="nrom">hachi</div><div class="nkj">八</div></div><div class="ncard"><div class="nnum">9</div><div class="nkana">きゅう／く</div><div class="nrom">kyū / ku</div><div class="nkj">九</div></div><div class="ncard"><div class="nnum">10</div><div class="nkana">じゅう</div><div class="nrom">jū</div><div class="nkj">十</div></div></div>
    <div class="gram"><p><b>4・7・9 有兩種唸法。</b>數數、報價錢、講數量時用 <b>よん・なな・きゅう</b>，這三個最保險；<b>し・しち・く</b> 主要出現在時間（4時＝よじ、7時＝しちじ、9時＝くじ）和固定詞裡。初學先記前面那組就好。</p></div>
  </section>
  <section>
    <h2>位數 UNITS <span>十・百・千・万</span></h2>
    <div class="ngrid"><div class="ncard"><div class="nnum">十</div><div class="nkana">じゅう</div><div class="nrom">jū</div><div class="nkj">10</div></div><div class="ncard"><div class="nnum">百</div><div class="nkana">ひゃく</div><div class="nrom">hyaku</div><div class="nkj">100</div></div><div class="ncard"><div class="nnum">千</div><div class="nkana">せん</div><div class="nrom">sen</div><div class="nkj">1,000</div></div><div class="ncard"><div class="nnum">万</div><div class="nkana">まん</div><div class="nrom">man</div><div class="nkj">10,000</div></div></div>
    <div class="gram">
      <p>日文數字是<b>「幾個這個位數」堆起來</b>的，跟中文一模一樣。二十＝に＋じゅう、三百＝さん＋ひゃく。中文怎麼拆，日文就怎麼唸。</p>
      <p>唯一不同的是<b>「一」的處理</b>：十、百、千 前面不用加 いち（100 就是 ひゃく，不說 いちひゃく），但<b>万一定要說 いちまん</b>。</p>
    </div>
  </section>
  <section>
    <h2>組合練習 <span>拆開來看就不難</span></h2>
    <div class="ngrid wide"><div class="ncard"><div class="nnum">11</div><div class="nkana">じゅういち</div><div class="nrom">jū ichi</div><div class="nkj">十＋一</div></div><div class="ncard"><div class="nnum">23</div><div class="nkana">にじゅうさん</div><div class="nrom">nijū san</div><div class="nkj">二×十＋三</div></div><div class="ncard"><div class="nnum">58</div><div class="nkana">ごじゅうはち</div><div class="nrom">gojū hachi</div><div class="nkj">五×十＋八</div></div><div class="ncard"><div class="nnum">350</div><div class="nkana">さんびゃくごじゅう</div><div class="nrom">sanbyaku gojū</div><div class="nkj">三×百＋五×十</div></div><div class="ncard"><div class="nnum">1,500</div><div class="nkana">せんごひゃく</div><div class="nrom">sen gohyaku</div><div class="nkj">千＋五×百</div></div><div class="ncard"><div class="nnum">12,345</div><div class="nkana">いちまんにせんさんびゃくよんじゅうご</div><div class="nrom">ichiman nisen sanbyaku yonjū go</div><div class="nkj">一萬＋二千＋三百＋四十＋五</div></div></div>
  </section>
  <section>
    <h2>發音變化 <span>這六個要特別記</span></h2>
    <div class="ngrid"><div class="ncard"><div class="nnum">300</div><div class="nkana">さんびゃく</div><div class="nrom">sanbyaku</div><div class="nkj">不是さんひゃく</div></div><div class="ncard"><div class="nnum">600</div><div class="nkana">ろっぴゃく</div><div class="nrom">roppyaku</div><div class="nkj">ろく→ろっ</div></div><div class="ncard"><div class="nnum">800</div><div class="nkana">はっぴゃく</div><div class="nrom">happyaku</div><div class="nkj">はち→はっ</div></div><div class="ncard"><div class="nnum">3,000</div><div class="nkana">さんぜん</div><div class="nrom">sanzen</div><div class="nkj">せん→ぜん</div></div><div class="ncard"><div class="nnum">8,000</div><div class="nkana">はっせん</div><div class="nrom">hassen</div><div class="nkj">はち→はっ</div></div><div class="ncard"><div class="nnum">10,000</div><div class="nkana">いちまん</div><div class="nrom">ichiman</div><div class="nkj">萬前面一定要加いち</div></div></div>
    <div class="gram"><p>規律是<b>「唸起來卡卡的就會變音」</b>：さん 後面的 ひゃく 濁化成 びゃく；ろく・はち 遇到 ひゃく・せん 會縮成促音 ろっ・はっ，後面的 ひ 再變成 ぴ。念幾次就會自然。</p></div>
  </section>
  <section>
    <h2>練習 PRACTICE <span>點卡片看唸法，按下一個換題</span></h2>
    <div class="panel">
      <div class="opt"><b>出題範圍</b><div class="chips">
        <label for="nr-1">1 – 10</label><label for="nr-2">11 – 99</label>
        <label for="nr-3">100 – 999</label><label for="nr-4">1000 – 9999</label>
        <label for="nr-5">1萬以上</label></div></div>
      <div class="opt"><b>題目面</b><div class="seg">
        <label for="nf-n">數字 → 唸法</label><label for="nf-k">唸法 → 數字</label></div></div>
    </div>
    <div class="sline"><div id="nslot"></div></div>
    <div class="tools sbtns">
      <button type="button" id="btn-nsay">🔊 唸這個數字</button>
      <button type="button" id="btn-nnext">下一個 →</button>
    </div>
  </section>
</div>

<div class="page" id="page-p">__PHRASES__</div>

<div class="page" id="page-w">
  <div class="panel">
    <div class="opt"><b>假名</b><div class="seg">
      <label for="wk-h">平假名 ひらがな</label><label for="wk-k">片假名 カタカナ</label></div></div>
    <div class="opt"><b>清音行</b><div class="chips">__WCHIPS_COL__</div></div>
    <div class="opt"><b>濁音行</b><div class="chips">__WCHIPS_DAK__</div></div>
    <div class="opt"><b>拗音行</b><div class="chips">__WCHIPS_YO__</div></div>
    <div class="opt"><b>橫的段</b><div class="chips">__WCHIPS_ROW__</div></div>
    <div class="opt"><b>題目面</b><div class="seg">
      <label for="wf-k">單字 → 意思</label><label for="wf-r">意思 → 單字</label></div></div>
    <div class="opt"><b>出題字形</b><div class="seg">
      <label for="wm-a">全部假名</label><label for="wm-j">有漢字用漢字</label></div></div>
    <div class="opt"><b>卡片順序</b><div class="seg">
      <label for="wo-n">原順序</label><label for="wo-v">反向</label></div>
      <div class="tools">
        <button type="button" id="btn-wshuffle">🔀 隨機洗牌</button>
        <button type="button" id="btn-wback">全部翻回</button>
        <button type="button" id="btn-wopen">全部翻開</button>
      </div></div>
  </div>
  __WDECK__
</div>

<div class="page" id="page-q">
  <div class="panel">
    <div class="opt"><b>假名</b><div class="seg">
      <label for="qk-h">平假名 ひらがな</label><label for="qk-k">片假名 カタカナ</label></div></div>
    <div class="opt"><b>選一直行</b><div class="chips">__CHIPS_COL__</div></div>
    <div class="opt"><b>選一橫行</b><div class="chips">__CHIPS_ROW__</div></div>
    <div class="opt"><b>題目面</b><div class="seg">
      <label for="qf-k">假名 → 拼音</label><label for="qf-r">拼音 → 假名</label></div></div>
    <div class="opt"><b>卡片順序</b><div class="seg">
      <label for="qo-n">原順序</label><label for="qo-v">反向</label></div>
      <div class="tools">
        <button type="button" id="btn-shuffle">🔀 隨機洗牌</button>
        <button type="button" id="btn-back">全部翻回</button>
        <button type="button" id="btn-open">全部翻開</button>
      </div></div>
  </div>
  __DECKS__
</div>
<p class="credit">筆順資料來源：<a href="http://kanjivg.tagaini.net">KanjiVG</a>（Ulrich Apel，CC BY-SA 3.0），本頁依相同條款釋出。發音使用裝置內建的日語語音；iPad／iPhone 可到「設定 → 輔助使用 → 朗讀內容 → 語音 → 日文」下載「進階」或「超高品質」語音，音質會明顯提升。</p>
</div>
__ZOOM__
<script>
(function(){
  function wire(sel, ids, orderIds){
    var deck=document.querySelector(sel);
    if(!deck) return;
    var cards=[].slice.call(deck.querySelectorAll('.fcard'));
    function clearOrder(){ cards.forEach(function(c){ c.style.order=''; }); }
    function flipAll(on){ deck.querySelectorAll('.fin').forEach(function(i){ i.checked=on; }); }
    var sh=document.getElementById(ids[0]);
    if(sh) sh.addEventListener('click',function(){
      var n=cards.length,p=[],i,j,t;
      for(i=0;i<n;i++) p.push(i);
      for(i=n-1;i>0;i--){ j=Math.floor(Math.random()*(i+1)); t=p[i]; p[i]=p[j]; p[j]=t; }
      cards.forEach(function(c,k){ c.style.order=p[k]; });
      flipAll(false);
    });
    var bk=document.getElementById(ids[1]);
    if(bk) bk.addEventListener('click',function(){ flipAll(false); });
    var op=document.getElementById(ids[2]);
    if(op) op.addEventListener('click',function(){ flipAll(true); });
    orderIds.forEach(function(id){
      var el=document.getElementById(id);
      if(el) el.addEventListener('change',clearOrder);
    });
  }
  wire('.deck', ['btn-shuffle','btn-back','btn-open'], ['qo-n','qo-v']);
  wire('.wdeck',['btn-wshuffle','btn-wback','btn-wopen'],['wo-n','wo-v']);
})();
</script>
<script>
/* ===== 句型練習 ===== */
var SPOOL=__SPOOL__;
(function(){
  var slot=document.getElementById('sslot');
  if(!slot) return;
  var keys=[].slice.call(document.querySelectorAll('input[id^="ss-"]'));
  var pool=[], idx=0;
  function esc(t){return (t||'').replace(/&/g,'&amp;').replace(/</g,'&lt;');}
  function build(){
    var sc=document.getElementById('sk-k').checked?'k':'h';
    var sel={};
    keys.forEach(function(k){ if(k.checked) sel[k.id.slice(3)]=1; });
    pool=(SPOOL[sc]||[]).filter(function(it){
      for(var i=0;i<it.sc.length;i++) if(sel[it.sc[i]]) return true;
      return false;
    });
    idx=0; render();
  }
  function render(){
    document.getElementById('szhw').textContent='＿＿＿';
    if(!pool.length){ slot.innerHTML='<div class="sempty">請至少選一行或一段</div>'; return; }
    var it=pool[idx%pool.length];
    var useKj=document.getElementById('sm-j').checked && it.kj;
    slot.innerHTML='<div class="scard"><span class="inner">'+
      '<span class="face fk"><span class="wfk">'+esc(useKj?it.kj:it.w)+'</span></span>'+
      '<span class="face fr"><span class="wans">'+esc(it.w)+'</span>'+
      '<span class="wrom">'+esc(it.rom)+'</span>'+
      (it.kj?'<span class="kj">'+esc(it.kj)+'</span>':'')+
      '<span class="wmean">'+esc(it.mean)+'</span>'+
      '<span class="wen">'+esc(it.en)+'</span></span></span></div>';
    var c=slot.querySelector('.scard');
    c.addEventListener('click',function(){
      c.classList.toggle('flip');
      document.getElementById('szhw').textContent=
        c.classList.contains('flip')?it.mean:'＿＿＿';
    });
  }
  document.getElementById('btn-snext').addEventListener('click',function(){
    if(pool.length){ idx=(idx+1)%pool.length; render(); }
  });
  document.getElementById('btn-ssay').addEventListener('click',function(){
    if(!pool.length) return;
    speak(pool[idx%pool.length].w+'が　すきです');
  });
  keys.concat([document.getElementById('sk-h'),document.getElementById('sk-k')])
      .forEach(function(el){ el.addEventListener('change',build); });
  ['sm-a','sm-j'].forEach(function(id){
    document.getElementById(id).addEventListener('change',render);
  });
  build();
})();
</script>
<script>
/* ===== 數字練習 ===== */

var N1K=['','いち','に','さん','よん','ご','ろく','なな','はち','きゅう'];
var N1R=['','ichi','ni','san','yon','go','roku','nana','hachi','kyū'];
var KANJI=['','一','二','三','四','五','六','七','八','九'];
function jpNum(n){
  if(n===0) return {k:'ゼロ',r:'zero',j:'零'};
  var k='',rp=[],j='';
  var man=Math.floor(n/10000), rest=n%10000;
  if(man){ k+=(man===1?'いち':N1K[man])+'まん'; rp.push((man===1?'ichi':N1R[man])+'man'); j+=(man===1?'一':KANJI[man])+'万'; }
  var sen=Math.floor(rest/1000); rest=rest%1000;
  if(sen){
    if(sen===1){k+='せん';rp.push('sen');}
    else if(sen===3){k+='さんぜん';rp.push('sanzen');}
    else if(sen===8){k+='はっせん';rp.push('hassen');}
    else {k+=N1K[sen]+'せん'; rp.push(N1R[sen]+'sen');}
    j+=(sen===1?'':KANJI[sen])+'千';
  }
  var hya=Math.floor(rest/100); rest=rest%100;
  if(hya){
    if(hya===1){k+='ひゃく';rp.push('hyaku');}
    else if(hya===3){k+='さんびゃく';rp.push('sanbyaku');}
    else if(hya===6){k+='ろっぴゃく';rp.push('roppyaku');}
    else if(hya===8){k+='はっぴゃく';rp.push('happyaku');}
    else {k+=N1K[hya]+'ひゃく'; rp.push(N1R[hya]+'hyaku');}
    j+=(hya===1?'':KANJI[hya])+'百';
  }
  var ju=Math.floor(rest/10), ich=rest%10;
  if(ju){
    if(ju===1){k+='じゅう';rp.push('jū');}
    else {k+=N1K[ju]+'じゅう'; rp.push(N1R[ju]+'jū');}
    j+=(ju===1?'':KANJI[ju])+'十';
  }
  if(ich){ k+=N1K[ich]; rp.push(N1R[ich]); j+=KANJI[ich]; }
  return {k:k,r:rp.join(' '),j:j};
}

(function(){
  var slot=document.getElementById('nslot');
  if(!slot) return;
  var RANGES={1:[1,10],2:[11,99],3:[100,999],4:[1000,9999],5:[10000,99999]};
  var cur=null;
  function pick(){
    var on=[];
    for(var i=1;i<=5;i++) if(document.getElementById('nr-'+i).checked) on.push(i);
    if(!on.length) return null;
    var r=RANGES[on[Math.floor(Math.random()*on.length)]];
    return r[0]+Math.floor(Math.random()*(r[1]-r[0]+1));
  }
  function render(){
    var n=pick();
    if(n===null){ slot.innerHTML='<div class="sempty">請至少選一個範圍</div>'; cur=null; return; }
    cur=n; var x=jpNum(n);
    var num='<span class="nbig">'+n.toLocaleString('en-US')+'</span>';
    var ans='<span class="nans">'+x.k+'</span><span class="wrom">'+x.r+'</span>'+
            '<span class="kj">'+x.j+'</span>';
    var qFirst=document.getElementById('nf-n').checked;
    slot.innerHTML='<div class="ncard2"><span class="inner">'+
      '<span class="face fk">'+(qFirst?num:ans)+'</span>'+
      '<span class="face fr">'+(qFirst?ans:num)+'</span></span></div>';
    var c=slot.querySelector('.ncard2');
    c.addEventListener('click',function(){ c.classList.toggle('flip'); });
  }
  document.getElementById('btn-nnext').addEventListener('click',render);
  document.getElementById('btn-nsay').addEventListener('click',function(){
    if(cur!==null) speak(jpNum(cur).k);
  });
  for(var i=1;i<=5;i++) document.getElementById('nr-'+i).addEventListener('change',render);
  ['nf-n','nf-k'].forEach(function(id){ document.getElementById(id).addEventListener('change',render); });
  render();
})();
</script>
<script>
var KVG=__KVG__;
(function(){
  document.body.classList.add('js');
  var auto=false;
  /* ---- 發音 ---- */
  var jaVoice=null;
  function rank(v){
    var n=(v.name||'').toLowerCase(), s=0;
    if(/premium|超/.test(n)) s+=6;
    if(/enhanced|neural|進階|拡張/.test(n)) s+=4;
    if(/kyoko|o-ren|oren|hattori|otoya|mizuki|nanami|ayumi/.test(n)) s+=2;
    if(v.localService) s+=1;
    return s;
  }
  function pickVoice(){
    if(!window.speechSynthesis) return;
    var vs=(speechSynthesis.getVoices()||[]).filter(function(v){ return /^ja/i.test(v.lang||''); });
    if(!vs.length) return;
    vs.sort(function(a,b){ return rank(b)-rank(a); });
    jaVoice=vs[0];
  }
  pickVoice();
  if(window.speechSynthesis && 'onvoiceschanged' in speechSynthesis){
    speechSynthesis.onvoiceschanged=pickVoice;
  }
  var primed=false;
  function prime(){
    if(primed||!window.speechSynthesis) return;
    primed=true;
    try{
      var u=new SpeechSynthesisUtterance(' ');
      u.volume=0; speechSynthesis.speak(u);
    }catch(e){}
  }
  document.addEventListener('touchend',prime,true);
  document.addEventListener('click',prime,true);
  function speak(t,slow){
    if(!window.speechSynthesis) return;
    try{
      if(!jaVoice) pickVoice();
      var u=new SpeechSynthesisUtterance(t);
      if(jaVoice) u.voice=jaVoice;
      u.lang='ja-JP'; u.pitch=1; u.volume=1; u.rate= slow?0.8:0.95;
      if(speechSynthesis.speaking||speechSynthesis.pending) speechSynthesis.cancel();
      speechSynthesis.resume();
      speechSynthesis.speak(u);   /* iOS 必須在點擊當下同步呼叫，不可延後 */
    }catch(e){}
  }
  /* ---- 筆順 ---- */
  var SMALLJP='\u3041\u3043\u3045\u3047\u3049\u3083\u3085\u3087\u308e\u3063\u30a1\u30a3\u30a5\u30a7\u30a9\u30e3\u30e5\u30e7\u30ee\u30c3';
  function strokeBox(ch){
    var ds=KVG[ch]; if(!ds) return null;
    var ns='http://www.w3.org/2000/svg';
    var svg=document.createElementNS(ns,'svg');
    svg.setAttribute('viewBox','0 0 109 109');
    if(SMALLJP.indexOf(ch)>=0) svg.setAttribute('class','smsvg');
    ds.forEach(function(d){
      var p=document.createElementNS(ns,'path');
      p.setAttribute('d',d); p.setAttribute('class','gh'); svg.appendChild(p);
    });
    var live=[];
    ds.forEach(function(d){
      var p=document.createElementNS(ns,'path');
      p.setAttribute('d',d); p.setAttribute('class','ln'); svg.appendChild(p); live.push(p);
    });
    return {svg:svg, paths:live};
  }
  function animate(paths){
    var delay=0;
    paths.forEach(function(p){
      var L; try{ L=p.getTotalLength(); }catch(e){ return; }
      if(!L) return;
      p.style.transition='none';
      p.style.strokeDasharray=L+' '+L;
      p.style.strokeDashoffset=L;
      var dur=Math.max(300,Math.min(1100,L*11));
      setTimeout(function(){
        p.style.transition='stroke-dashoffset '+dur+'ms linear';
        p.style.strokeDashoffset=0;
      },delay+30);
      delay+=dur+180;
    });
  }
  function play(v){
    var ch=v.getAttribute('data-k'), box=v.querySelector('.zsvg');
    if(box.dataset.built==='1'){
      animate([].slice.call(box.querySelectorAll('svg > path.ln'))); return;
    }
    box.innerHTML=''; var all=[];
    [].forEach.call(ch,function(c){
      var b=strokeBox(c);
      if(b){ box.appendChild(b.svg); all=all.concat(b.paths); }
    });
    if(!all.length) return;
    box.dataset.built='1';
    v.classList.add('stroke');
    animate(all);
  }
  document.querySelectorAll('.zview').forEach(function(v){
    var ch=v.getAttribute('data-k');
    v.querySelector('.zsp').addEventListener('click',function(){ speak(ch,true); });
    v.querySelector('.b-stk').addEventListener('click',function(){ play(v); });
    var mt=v.querySelector('.mtog');
    if(mt) mt.addEventListener('click',function(){ mt.parentNode.classList.toggle('show'); });
    v.querySelectorAll('.w').forEach(function(b){
      b.addEventListener('click',function(){
        b.classList.toggle('show'); speak(b.getAttribute('data-w'));
      });
    });
  });
  document.querySelectorAll('.p').forEach(function(b){
    b.addEventListener('click',function(){
      b.classList.toggle('show'); speak(b.getAttribute('data-w'));
    });
  });
  document.querySelectorAll('.zin').forEach(function(inp){
    inp.addEventListener('change',function(){
      var v=inp.nextElementSibling;
      v.querySelectorAll('.show').forEach(function(e){ e.classList.remove('show'); });
      if(inp.checked){ play(v); if(auto) speak(v.getAttribute('data-k'),true); }
    });
  });
})();
</script>
</body>
</html>"""

## 14. 組裝並寫出 index.html

In [14]:
HTML = (HTML.replace("__GEN__",GEN).replace("__RADIOS__","".join(radios))
  .replace("__CHART_H__",CH_H).replace("__PHRASES__",phrases_html()).replace("__CHART_K__",CH_K)
  .replace("__CHIPS_COL__",chips_col).replace("__CHIPS_ROW__",chips_row)
  .replace("__DECKS__",deckhtml).replace("__WDECK__",wdeck_html()).replace("__WCHIPS_COL__",wchips_col).replace("__WCHIPS_ROW__",wchips_row)
  .replace("__WCHIPS_DAK__",wchips_dak).replace("__WCHIPS_YO__",wchips_yo)
  .replace("__SCHIPS_COL__",wchips_col.replace('for="ws-','for="ss-'))
  .replace("__SCHIPS_ROW__",wchips_row.replace('for="ws-','for="ss-'))
  .replace("__SCHIPS_DAK__",wchips_dak.replace('for="ws-','for="ss-'))
  .replace("__SCHIPS_YO__",wchips_yo.replace('for="ws-','for="ss-'))
  .replace("__SPOOL__",json.dumps(spool(),ensure_ascii=False,separators=(',',':'))).replace("__ZOOM__",zoomhtml).replace("__KVG__",KVGJSON))

OUT.write_text(HTML, encoding="utf-8")
print("cells:",zc[0],"cards:",fc[0],"bytes:",len(HTML.encode()))

cells: 208 cards: 142 bytes: 595431


## 15. 驗證

簡單檢查產出的檔案是否完整。

In [15]:
import re

html = OUT.read_text(encoding="utf-8")
left = set(re.findall(r"__[A-Z][A-Z_]*__", html))
checks = {
    "放大字卡": html.count('class="zview"'),
    "字卡測驗牌": html.count('class="fcard sc-'),
    "單字測驗卡": html.count('class="fcard wq'),
    "常用短句": html.count('class="p" data-w'),
    "筆順資料": len(json.loads(KVG_PATH.read_text(encoding="utf-8"))),
    "檔案大小 (KB)": round(len(html.encode()) / 1024),
}
for k, v in checks.items():
    print(f"{k:<14}{v}")

assert not left, f"還有佔位符沒被替換：{left}"
assert checks["放大字卡"] > 200 and checks["單字測驗卡"] > 100, "內容數量異常"
print("\n檢查通過，index.html 已更新，可以 commit 了。")

放大字卡          208
字卡測驗牌         142
單字測驗卡         330
常用短句          15
筆順資料          148
檔案大小 (KB)     581

檢查通過，index.html 已更新，可以 commit 了。
